In [ ]:
# Installation des bibliotheques necessaires

# Pour lire et manipuler des fichiers PDF
!pip install PyPDF2
!pip install PyMuPDF

!pip install langchain_text_splitters # Pour découper les textes en segments utilisables par les LLMs
!pip install sentence_transformers # Pour générer des embeddings (représentations vectorielles des textes)
!pip install faiss-cpu # Pour la recherche de similarité dans des vecteurs
!pip install langchain langchain-community # Pour utiliser les outils de la bibliothèque LangChain

# Pour utiliser le service d'IA de Groq avec LangChain
!pip install langchain-groq
!pip install groq

!pip install flask pyngrok# Pour créer une API web locale (Flask) et l'exposer via ngrok

# Importation des bibliothèques necessaires

from transformers import AutoTokenizer, AutoModel # Import de modèles pré-entraînés et de tokenizers pour NLP
from sentence_transformers import SentenceTransformer, models # Import du modèle SentenceTransformer pour générer des embeddings
import faiss # Import de Faiss pour effectuer des recherches de similarité vectorielle
import numpy as np

# Import des modules Flask pour créer une application web
from flask import Flask
from flask import Flask, render_template, request, redirect, url_for

In [ ]:
#Nettoyage des notes qui se trouvent en en gauche de chaque page , selon leur couleur et tailles en utilisant fitz de PyMuPDF
import fitz  # PyMuPDF

def is_note(span):
    return span["size"] < 10 or span["color"] in [8421504, 12632256]

def extract_clean_text(pdf_path):
    doc = fitz.open(pdf_path)
    cleaned_text = ""
    for page in doc:
        blocks = page.get_text("dict")["blocks"]
        for b in blocks:
            for line in b.get("lines", []):
                for span in line.get("spans", []):
                    if not is_note(span):
                        cleaned_text += span["text"] + " "
                cleaned_text += "\n"
    return cleaned_text

In [ ]:
#Prétraitement et segmentation du texte en utilisant  Regular expressions
import re
def pretraitment_de_text(text):
  text = re.sub(r'â€™', "", text)  # Supprime les caractères mal encodés
  text = re.sub(r'[^\x00-\x7F]+', '', text)  # Supprimer les caractères non ASCII
  text=re.sub(r"Page \d","",text) #Suppressiondes numéros des pages
  # Suppression des métadonnées, références au site, titres redondants, contenu promotionnel, table des matières et index
  text = re.sub(r'English Translation.*?distribution on.', '', text)
  text=re.sub(r"constituteproject.org",'',text,flags=re.IGNORECASE)
  text=re.sub(r"constituteprojectorg",'',text,flags=re.IGNORECASE)
  text=text.strip()
  text=re.sub(r"PDF generated: 23 Nov 2017, 15:19","",text).strip()
  text=re.sub(r"Morocco 2011","",text).strip()
  text=re.sub(r"Morocco's Constitution of 2011","",text).strip()
  text=re.sub(r"with content generously.*?World Constitutions Illustrated","",text).strip()
  text = re.sub(r"Table of contents.*?Title XIV: Transitory and Final Provisions", "", text, flags=re.DOTALL)
  text = re.sub(r"Topic index \n.*?Veto override procedure","",text,flags=re.DOTALL)
  text = text.strip()
  return text

In [ ]:
#Découpage selon des articles
import re

def split_constitution_by_articles(text):
    article_pattern = r"(Article\s+\d+\s+\n)" # Motif pour détecter les titres d'articles (ex : "Article 5\n")
    split_text = re.split(article_pattern, text)
    # Contenu de l'article 1 ajouté manuellement
    content="Morocco is a constitutional, democratic, parliamentary and social Monarchy.The constitutional regime of the Kingdom is founded on the separation, the balanceand the collaboration of the powers, as well as on participative democracy of [the] citizen, and the principles of good governance and of the correlation between the responsibility for and the rendering of accounts. The Nation relies for its collective life on the federative constants [constantes federatrices], on the occurrence of moderate Muslim religion, [on] the national unity of its multiple components [affluents], [on] the constitutional monarchy and [on] democratic choice. The territorial organization of the Kingdom is decentralized. It is founded on an advanced regionalization."
    # Listes pour stocker les articles sous forme de texte brut et de dictionnaire
    articles = []
    articles2=[]
    articles.append(f"Article 1\n{content}") # Ajout manuel de l'article 1
    articles2.append({"title": "Article 1", "content": content})
    for i in range(1, len(split_text), 2):
        title = split_text[i].strip() # Titre de l'article
        content = split_text[i+1].strip() # Contenu associé à ce titre
        articles.append(f"{title}\n{content}") # Format texte
        articles2.append({"title": title, "content": content})  # Format dict

    return articles,articles2

In [ ]:
def split_articles_chunks(articles):
  all_chunks = []
  for article in articles:
    all_chunks.append(article)
  return all_chunks

In [ ]:
text=extract_clean_text("Morocco Constitution 2011-English.pdf") #Extraction du texte
text=pretraitment_de_text(text) #Nettoyage du texte
articles,articles2=split_constitution_by_articles(text) #Decoupage en articles
all_chunks=split_articles_chunks(articles)

# Affichage des articles extraits
for i, article in enumerate(articles):
    print(f"--- Article {i+1} ---\n{article}\n{'-'*50}")

In [ ]:
#Sauvegarde des articles dans un fichier JSON
import json
with open("articles2.json", "w", encoding="utf-8") as f:
    json.dump(articles2, f, ensure_ascii=False, indent=4)

In [ ]:
def Embedding(articles):
  word_embedding_model = models.Transformer('nlpaueb/legal-bert-base-uncased') # Chargement du modèle Legal-BERT depuis Hugging Face (spécialisé pour les textes juridiques)
  pooling_model = models.Pooling(word_embedding_model.get_word_embedding_dimension()) # Pooling : regroupe les vecteur de mots

  embedding_model = SentenceTransformer(modules=[word_embedding_model, pooling_model]) # Creation du modèle SentenceTransformer avec les deux modules

  embeddings = embedding_model.encode(articles)# Génération des embiddings pour chaque article
  return embeddings # Retourne une liste de vecteurs

In [ ]:
#base de données vectoriel avec FAISS
import faiss
import numpy as np
def base_donnees_vectoriel(embeddings):

  embeddings = np.array(embeddings).astype('float32') # Conversion des embeddings en tableau NumPy (type float32 requis par FAISS)

  index = faiss.IndexFlatL2(embeddings.shape[1]) # Création d’un index FAISS basé sur la distance L2 (euclidienne)
  index.add(embeddings) # Ajout des vecteurs à l’index
  return index # Retourne l’index vectoriel prêt pour la recherche

In [ ]:
def generer_reponse_chatgroq(query, passages):
    from langchain_groq.chat_models import ChatGroq
    from langchain_core.messages import HumanMessage, SystemMessage

    context = "\n\n".join(passages) # Fusionner les extraits (passages) en un seul contexte

    # Initialisation du modèle Groq avec Gemma 2
    llm = ChatGroq(model_name="gemma2-9b-it",
                   api_key="TON_API_KEY", # Remplacez par votre clé API Groq
                   temperature=0.0,# Réponses déterministes
                   max_tokens=500# Limite de la longueur de la réponse
                   )
    # Définition du comportement du système
    messages = [
    SystemMessage(content="You are an AI legal assistant trained to answer questions strictly based on excerpts from the Moroccan Constitution and you are not a licensed legal advisor or lawyer. Respond only based on the excerpts provided. When you answer, cite the articles used and, if relevant, include a quote from the article to support your response. Be clear, structured, and precise."),
    HumanMessage(content=f"Here are the excerpts from the Constitution:\n{context}\n\nQuestion: {query}\n\nRespond based only on these excerpts.")]


    response = llm.invoke(messages) #Envoi de la requête au modele
    return response.content #Retourne uniquement le texte généré



In [ ]:
#Recherche des similaritées

def Recherche(query,index):
  query_embedding=Embedding([query])# Encoder la question en vecteur d'embedding
  query_embedding = np.array(query_embedding).astype('float32')
  k = 7  # nombre de résultats à récupérer
  distances, indices = index.search(query_embedding, k)#Recuperation des articles correspondants aux indices retournés
  relevant_passages = [articles[i] for i in indices[0]]#Récupérer les passages les plus pertinents pour la question
  reponse=generer_reponse_chatgroq(query,relevant_passages) #Generation de la reponse
  return reponse



In [ ]:
# Création du dossier 'templates' et génération du fichier HTML pour l'interface web du chatbot

import os # module os pour gérer les opérations système (création de dossier, etc.)

os.makedirs("templates", exist_ok=True)

html_code = '''<!DOCTYPE html>
<html lang="fr">
<head>
    <meta charset="UTF-8" />
    <title>Chatbot Constitution Marocaine</title>
    <style>
        /* tes styles et animations fadeInLeft / fadeInRight ici */
        @keyframes fadeInLeft {
            from {
                opacity: 0;
                transform: translateX(-30px);
            }
            to {
                opacity: 1;
                transform: translateX(0);
            }
        }

        @keyframes fadeInRight {
            from {
                opacity: 0;
                transform: translateX(30px);
            }
            to {
                opacity: 1;
                transform: translateX(0);
            }
        }

        body {
            margin: 0;
            font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
            background-color: #f5f0e6;
            display: flex;
            flex-direction: column;
            height: 100vh;
            color: #3b3a30;
        }

        .chat-container {
    flex: 1;
    padding: 40px 20px; /* ↑ augmente l'espacement vertical */
    margin: 0 auto;
    max-width: 800px;
    overflow-y: auto;
    display: flex;
    flex-direction: column;
    align-items: center;
}


        .message {
            max-width: 70%;
            padding: 14px 20px;
            margin: 8px 0;
            border-radius: 20px;
            font-size: 16px;
            line-height: 1.4;
            box-shadow: 0 4px 10px rgba(0,0,0,0.1);
            word-wrap: break-word;
            opacity: 1;
        }

        /* Messages sans animation */
        .user-message {
            background-color: #cde9ff;
            align-self: flex-end;
            box-shadow: 0 4px 12px rgba(60,130,230,0.3);
            margin-left: auto;
            margin-right: 20px;
            color: #1a2737;
        }

        .bot-message {
            background-color: #fff8e7;
            align-self: flex-start;
            box-shadow: 0 4px 12px rgba(220,180,90,0.3);
            margin-right: auto;
            margin-left: 20px;
            color: #4a462c;
        }

        /* Animation sur le dernier message */
        .user-message.last {
            animation-name: fadeInRight;
            animation-duration: 0.4s;
            animation-fill-mode: forwards;
            opacity: 0;
        }

        .bot-message.last {
            animation-name: fadeInLeft;
            animation-duration: 0.4s;
            animation-fill-mode: forwards;
            opacity: 0;
        }

        .form-container {
    display: flex;
    justify-content: center;
    padding: 25px 20px; /* ↑ plus d'espacement vertical */
    background-color: #fdf9f1;
    border-top: 1px solid #ddd6c8;
}


        input[type="text"] {
            flex: 1;
            padding: 14px 20px;
            border-radius: 25px;
            border: 1.5px solid #d8d0b5;
            font-size: 16px;
            outline: none;
            transition: border-color 0.3s ease;
            color: #3b3a30;
            background-color: #fefcf7;
        }

        input[type="text"]:focus {
            border-color: #a99c67;
            background-color: #fffde7;
        }

        button {
            margin-left: 15px;
            padding: 14px 25px;
            background-color: #a99c67;
            color: #fefcf7;
            border: none;
            border-radius: 25px;
            font-size: 16px;
            cursor: pointer;
            transition: background-color 0.3s ease;
        }

        button:hover {
            background-color: #827a4c;
        }

        @media (max-width: 600px) {
            .chat-container {
                padding: 15px 20px;
                align-items: stretch;
            }

            .message {
                max-width: 90%;
                border-radius: 16px;
            }

            .user-message {
                margin-right: 20px;
                margin-left: auto;
            }

            .bot-message {
                margin-left: 20px;
                margin-right: auto;
            }

            .form-container {
                padding: 10px 20px;
            }

            input[type="text"], button {
                font-size: 14px;
                padding: 12px 15px;
            }

            button {
                margin-left: 10px;
                border-radius: 20px;
            }
        }
    </style>
</head>
<body>
<header style="text-align:center; padding: 20px; background-color: #f2e9d8; color: #3b3a30;">
    <h1 style="margin: 0; font-size: 28px;">Chatbot Constitution Marocaine</h1>
    <p style="font-size: 16px; margin-top: 8px;">Posez vos questions sur les articles, les droits et les institutions du Royaume</p>
</header>

    <div class="chat-container" id="chat-box">
        {% for question, response in chat_history[:-1] %}
            <div class="message user-message">{{ question }}</div>
            <div class="message bot-message">{{ response }}</div>
        {% endfor %}

        {% if chat_history %}
            <div class="message user-message last">{{ chat_history[-1][0] }}</div>
            <div class="message bot-message last" id="typing-bot-message"></div>
        {% endif %}
    </div>

    <div class="form-container">
        <form method="post" style="display: flex; width: 100%; max-width: 800px; margin: 0 auto;">
            <input type="text" name="query" placeholder="Posez votre question sur la Constitution marocaine..." required autocomplete="off" autofocus>
            <button type="submit">Envoyer</button>
        </form>
    </div>

    <script>
        const chatBox = document.getElementById("chat-box");
        chatBox.scrollTop = chatBox.scrollHeight;

        const lastBotResponse = {{ last_bot_response|tojson|safe }};
        const typingElement = document.getElementById("typing-bot-message");

        if (lastBotResponse && typingElement) {
            const words = lastBotResponse.split(' ');
            let index = 0;
            typingElement.textContent = '';
            typingElement.style.opacity = '1';

            function typeWord() {
                if (index < words.length) {
                    typingElement.textContent += (index === 0 ? '' : ' ') + words[index];
                    index++;
                    chatBox.scrollTop = chatBox.scrollHeight;
                    setTimeout(typeWord, 200);
                }
            }
            typeWord();
        }
    </script>

</body>
</html>
'''

with open("templates/chat.html", "w", encoding="utf-8") as f:
    f.write(html_code)


In [ ]:
from flask import Flask, render_template, request
from pyngrok import ngrok,conf

# Authentification ngrok
conf.get_default().auth_token = "NGROK_AUTH_TOKEN"

# Initialisation d'une liste globale pour conserver l'historique des échanges de chat
chat_history = []

# App Flask
app = Flask("ISC")

@app.route('/', methods=['GET', 'POST'])
def chat():
    global chat_history
    last_bot_response = None # Variable pour stocker la dernière réponse du bot
    if request.method == 'POST':
        user_query = request.form['query'] # Récupération de la requête utilisateur depuis le formulaire
        bot_response = Recherche(user_query, index) #Appel de la fonction Recherhce

        if hasattr(bot_response, 'content'):
          bot_response = bot_response.content

        chat_history.append((user_query, bot_response)) # Ajout du couple (question utilisateur, réponse bot) dans l'historique du chat
        last_bot_response = bot_response # Mise à jour de la dernière réponse du bot
    return render_template('chat.html', chat_history=chat_history, last_bot_response=last_bot_response)# génère et renvoie la page web chat.html en y intégrant les données chat_history et last_bot_response

# Tunnel ngrok
# Ouverture d'un tunnel ngrok pour exposer localement le serveur Flask sur Internet
public_url = ngrok.connect(5000)
print("Ouvre cette URL dans un nouvel onglet :", public_url)

app.run(port=5000, host='0.0.0.0') # Démarrage sur le port 5000 accessible depuis toutes les interfaces réseau
